# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AIz') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash-lite'
# openai = OpenAI()
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [8]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevent links")
    return links

select_relevant_links("https://edwarddonner.com")
select_relevant_links("https://huggingface.co")
    

Found 3 relevent links
Found 7 relevent links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'company', 'url': 'https://huggingface.co/huggingface'}]}

In [9]:
select_relevant_links("https://edwarddonner.com")

Found 2 relevent links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash-lite
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'main page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 11 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-397B-A17B
Updated
3 days ago
•
218k
•
892
nvidia/personaplex-7b-v1
Updated
7 days ago
•
540k
•
2.15k
Nanbeige/Nanbeige4.1-3B
Updated
1 day ago
•
154k
•
717
zai-org/GLM-5
Updated
9 days ago
•
178k
•
1.43k
MiniMaxAI/MiniMax-M2.5
Updated
7 days ago
•
191k
•
858
Browse 2M+ models
Spaces
Running
on
Zero
MCP
864
Wan2.2 14B Preview
🐌
864
generate a video from an image with a text prompt
Running
Fea

In [ ]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 15 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nGet started with Inference in seconds 🚀\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-397B-A17B\nUpdated\n3 days ago\n•\n218k\n•\n892\nnvidia/personaplex-7b-v1\nUpdated\n7 days ago\n•\n540k\n•\n2.15k\nNanbeige/Nanbeige4.1-3B\nUpdated\n1 day ago\n•\n154k\n•\n717\nzai-org/GLM-5\nUpdated\n9 days ago\n•\n178k\n•\n1.43k\nMiniMaxAI/MiniMax-M2.5\

In [23]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 11 relevant links


## Hugging Face: Where AI Gets Its Hugs (and Does Amazing Things)

So, you've heard of AI, right? That fancy stuff that's supposedly taking over the world. Well, at Hugging Face, we're not building Skynet (yet). We're building the *community* that's building the future of AI. Think of us as the friendly neighborhood coffee shop, but instead of artisanal lattes, we serve up mind-boggling models, datasets that'll make your data dreams come true, and "Spaces" where cool AI apps hang out.

### What's the Deal Here?

Imagine a place where the brightest minds (and maybe a few caffeinated geniuses) in machine learning come to:

*   **Discover:** We've got over 2 million models. That's more models than you can shake a Transformer at! Whether you need something to write the next great novel, generate a video of a cat riding a unicorn, or translate Klingon to English, we've probably got it.
*   **Collaborate:** Got a brilliant idea? Want to fine-tune that existing model? Our platform is like a digital playground for ML enthusiasts. Share your work, get feedback, and maybe even find your AI soulmate.
*   **Build:** We provide the tools, the community, and the… well, the *hugs*… to help you accelerate your AI projects. From text to image, audio to 3D, we're exploring all the modalities your AI heart could desire.

### Our Vibe?

We're all about open source, community, and moving fast. We're the ones who brought GGML and llama.cpp into the Hugging Face family, so you know we're not afraid to get our hands (or our code) dirty. Our culture is built on collaboration and innovation, with a healthy dose of enthusiasm for all things AI. We believe that building the future of AI should be fun, accessible, and a little bit wild.

### Who are our Customers?

Basically, anyone who wants to do cool stuff with AI. That includes researchers, developers, startups, and even large enterprises looking to leverage the power of machine learning. If you're building an AI-powered app, working on a groundbreaking research paper, or just want to experiment with the latest AI tech, you'll feel right at home here.

### Want to Join the Party? (Careers)

Are you tired of explaining what a neural network is to your aunt Mildred? Do you dream in Python and find joy in a perfectly tuned hyperparameter? Then you might just be one of us! We're always looking for passionate individuals to join our growing team. Come help us build the future of AI, one hug at a time. Check out our careers page – you might just find your perfect AI match.

**In short: Hugging Face is the place to be if you love AI, collaboration, and the thrill of building something amazing.** Come on over, browse our models, try out an app, and let's build the future together!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 23 relevant links


## HuggingFace: Where AI Gets Real (and Occasionally a Little Weird)

So, you've heard of AI, right? That thing that's either going to solve all our problems or turn us into paperclips? Well, at HuggingFace, we're busy building the "solving problems" part. Think of us as the ultimate playground for anyone who loves machine learning, wants to build cool AI stuff, or just enjoys a good ol' fashioned AI collaboration.

**What's HuggingFace?**

Imagine a giant, slightly chaotic, but incredibly brilliant party for AI. That's us. We're a community and a platform where the brightest minds (and some folks who just really like cats in AI-generated images) come together. We've got:

*   **2 Million+ Models:** That's right, MILLION. From tiny AI helpers to behemoths that can write poetry or code. We're pretty sure there's a model here that can do that weird dance you've been practicing.
*   **1 Million+ Applications (Spaces):** Want to see an AI generate a video from a text prompt? Or maybe have a chat with an AI that *might* be sentient? Our Spaces are where the magic (and occasional glitches) happen. We've got demos so cool, they're practically hugging you.
*   **500k+ Datasets:** Because AI needs food, and we've got more data than a squirrel hoarding nuts for winter.

**Our Vibe: Open, Collaborative, and Fueled by Code (and Coffee)**

We're all about open-source, collaboration, and making AI accessible. Our culture? Think less stuffy corporate boardroom, more energetic hackathon. We're excited about what's next, love to experiment, and aren't afraid to admit when a model does something hilariously unexpected. We even have courses on things like "smol" AI and how to let a lobster run your robot. No, seriously.

**Who are our Customers?**

Anyone who wants to play with, build, or deploy AI! From individual researchers and hobbyists to enterprise giants looking to supercharge their AI game. If you're curious about AI, you're already one of us.

**Fancy a Career Here?**

Are you passionate about AI? Do you dream in code? Can you explain complex AI concepts using cat analogies? If you answered "yes" (or even a hesitant "meow"), then you might be a perfect fit. We're building the future, and we need brilliant people to help us do it. Plus, we have excellent documentation and courses to help you learn along the way. Come join the party – just try not to break the AI.

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 13 relevant links


## Hugging Face: Where AI Gets a Hug (and Your Brain Gets a Workout!)

Tired of AI that feels like it's judging your life choices? Welcome to Hugging Face, the *most* enthusiastic AI community on the internet. We're building the future, one giggling, learning, and occasionally glitchy model at a time.

**What's the Big Idea?**

Imagine a digital playground overflowing with brilliant minds and even brighter algorithms. That's us! We're the platform where the machine learning community comes to play, share, and collaborate on everything AI. Think of it as GitHub, but instead of code for your grandma's toaster, it's code for robots that can write poetry, paint like Picasso (or at least *try* to), and chat your ear off about the latest AI trends.

**Our Stacks of Stuff:**

*   **Models:** We've got over 2 million of them! From "can it write a Shakespearean sonnet about pizza?" to "can it generate a video of a cat riding a unicycle on the moon?", we've got a model for that. Trending this week? Some fascinating stuff like Qwen/Qwen3.5-397B-A17B (sounds fancy, probably is) and nvidia/personaplex-7b-v1 (might help you find your long-lost twin… or just generate a spooky avatar).
*   **Datasets:** Because AI needs food, and our AI is *hangry*. We've got datasets for everything from your basic OCR needs to… well, honestly, some of these datasets are so niche, we're not entirely sure what they're for, but somebody out there is probably thrilled!
*   **Spaces:** This is where the magic happens, or at least where the AI tries its best. Generate videos from images, whip up some custom speech, or edit your photos with AI that's probably better at it than you are.

**Why Hug Us?**

*   **For the Community:** Join thousands of AI enthusiasts, researchers, and hobbyists who are genuinely excited about what's next. We're less "corporate overlords" and more "slightly eccentric AI aunts and uncles."
*   **For the Future:** Be part of something big. Whether you're a seasoned pro or just curious about how to make your cat an internet celebrity using AI, you'll find your tribe here.
*   **For the Fun:** AI can be serious business, but we believe it should also be ridiculously fun. We're constantly innovating, experimenting, and occasionally setting off digital fireworks (metaphorically, of course… mostly).

**Got AI Talent to Share? We Want You!**

We're a fast-growing company, and we're always looking for passionate individuals who want to contribute to the AI revolution. If you're a coder, a data wizard, a prompt engineer extraordinaire, or just someone who can explain AI to your grandma without making her cry, we might have a spot for you. Come build the future with us!

**So, what are you waiting for? Come on over to Hugging Face. We promise it's a lot less awkward than hugging a robot.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>